# The final paper and its code

A compact companion to **Simulating Game Theory and Strategic Interactions
Using Quantum Computing**, by Esther Cui. This walkthrough connects the
manuscript to the original notebooks and checks the circuit behavior directly.

**Finding:** the original grids and most Schelling rates are traceable. The
cooperative PD Nash interpretation fails a unilateral-deviation check; some
QAOA summaries, sampling budgets, and runtime descriptions also need correction.
See the [page-by-page comparison](../docs/paper-comparison.md).

The paper's reported values and fresh calculations are kept separate. No old
random seed, missing measurement counts, or external pickle is reconstructed.

## 1. Setup

From the repository root, run `python -m pip install -e '.[notebook,verify]'`.
This supplies the notebook tools and Qiskit used below. The current checks use
exact statevector probabilities. Aer, statsmodels, a QPU, and cloud credentials
are not needed. The unfinished historical z-test is not treated as evidence.

The installed package provides the algorithms; the local JSON source record
holds the manuscript transcription and archived output references.

In [1]:
import json
from itertools import product
from math import pi
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

from quantum_pd import expected_payoff
from quantum_pd.experiments import (
    classical_match,
    qaoa_outcomes,
    schelling_four_spots,
    schelling_probabilities,
)

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
reported = json.loads((root / "results/paper-reported.json").read_text())
print(reported["manuscript"]["title"])
print("PDF SHA-256:", reported["manuscript"]["sha256"])
print("Evaluation: current Qiskit, exact probabilities; no historical shot replay.")

Simulating Game Theory and Strategic Interactions Using Quantum Computing
PDF SHA-256: 46ae001c57f1bb9d275db0be4eb8daaa36029cf83efdd187f9d9e2f2f1d6fcdc
Evaluation: current Qiskit, exact probabilities; no historical shot replay.


## 2. Trace the reported PD and QAOA payoffs

These are manuscript values from Table 2, page 6, not the output of the current
solver. For k > 2 the original classical/QAOA code sums pairwise payoffs, while
the original circuit Nash code uses a collective payoff rule.

In [2]:
rows = [
    "| k | Paper classical payoff/player | Paper QAOA payoff/player | Paper circuit payoffs |",
    "|---:|---:|---:|---|",
]
for result in reported["table_2_page_6"]:
    rows.append(
        f"| {result['players']} | {result['classical_payoff_per_player']:.2f} | "
        f"{result['qaoa_payoff_per_player']:.2f} | {result['ewl_payoffs']} |"
    )
display(Markdown("\n".join(rows)))

| k | Paper classical payoff/player | Paper QAOA payoff/player | Paper circuit payoffs |
|---:|---:|---:|---|
| 2 | 1.00 | 2.50 | [3.0, 3.0] |
| 3 | 1.00 | 4.68 | [0.0, 5.0, 0.0] |
| 4 | 1.00 | 6.73 | [3.0, 3.0, 3.0, 3.0] |

## 3. Test the claimed Nash profiles

The defining test is whether any player can improve by changing only their own
strategy. For the original circuit, theta = 0 is C and theta = pi is D; phase
choices do not change probabilities. Each calculation below uses the original
collective payoff rule and the corrected player ordering.

In [3]:
rows = [
    "| k | Reported actions | Player 0 payoff | After player 0 defects | Gain |",
    "|---:|---|---:|---:|---:|",
]
for players, actions in [(2, [0, 0]), (3, [0, 1, 0]), (4, [0, 0, 0, 0])]:
    angles = [(pi * action, 0) for action in actions]
    before = expected_payoff(angles)[0]
    changed = angles.copy()
    changed[0] = (pi, 0)
    after = expected_payoff(changed)[0]
    assert after - before > 1e-3
    rows.append(f"| {players} | {actions} | {before:.0f} | {after:.0f} | +{after - before:.0f} |")
display(Markdown("\n".join(rows)))

| k | Reported actions | Player 0 payoff | After player 0 defects | Gain |
|---:|---|---:|---:|---:|
| 2 | [0, 0] | 3 | 5 | +2 |
| 3 | [0, 1, 0] | 0 | 5 | +5 |
| 4 | [0, 0, 0, 0] | 3 | 5 | +2 |

Each positive gain rules out the reported profile as a Nash equilibrium.
The full corrected searches find all-D at both the paper's 1e-3 tolerance and
the package's default 1e-10 tolerance. The
[PD walkthrough](01_prisoners_dilemma.ipynb) and
[computed checks](../results/paper-checks.json) show those searches.

## 4. Replay the QAOA heatmap domain

The original k = 3 heatmap uses seven gamma values in [0, pi/2] and seven beta
values in [0, pi]. Its code samples 2,048 shots. Here we evaluate the same grid
without sampling; this is not a repeat of the unseeded optimizer run.

In [4]:
grid_results = [
    qaoa_outcomes(3, gamma, beta)
    for gamma, beta in product(np.linspace(0, pi / 2, 7), np.linspace(0, pi, 7))
]
maximum = max(row["mean_payoff"] for row in grid_results)
assert np.isclose(maximum, 4.59375)
assert all(np.isclose(row["cooperation_rate"], 0.5) for row in grid_results)
print(f"Exact k = 3 grid maximum payoff/player: {maximum:.5f}")
print("Expected cooperation throughout this symmetric ansatz: 50%")
print("Pairwise classical all-D baseline at k = 3: 2 per player")

Exact k = 3 grid maximum payoff/player: 4.59375
Expected cooperation throughout this symmetric ansatz: 50%
Pairwise classical all-D baseline at k = 3: 2 per player


The paper's heatmap near 4.6 is consistent in scale with this calculation.
Its Table 2 value of 4.68 does not appear in the saved optimizer vector
(4.56, 4.58, 4.72); historical raw counts and seeds are unavailable. The 67%
cooperation entry also differs from the expected fraction of cooperating players,
which is 50% under this bit-complement-symmetric circuit.

## 5. Check the Schelling circuits

Without the inverse entangler, the Bell-pair match probability is
`cos²((theta_A - theta_B)/2)`. The inverse variant gives matching bitstrings
for these rotations, but includes an additional joint two-qubit operation.
These are distinct protocols.

In [5]:
rows = ["| Angles (A, B) | Without inverse | With inverse |", "|---|---:|---:|"]
for label, a, b in [
    ("0, 0", 0, 0),
    ("pi/6, pi/6", pi / 6, pi / 6),
    ("pi/6, 0", pi / 6, 0),
    ("pi/2, 0", pi / 2, 0),
]:
    probabilities = schelling_probabilities(a, b)
    raw_match = probabilities[0] + probabilities[3]
    decoded = schelling_probabilities(a, b, undo=True)
    decoded_match = decoded[0] + decoded[3]
    assert np.isclose(raw_match, np.cos((a - b) / 2) ** 2)
    assert np.isclose(decoded_match, 1)
    rows.append(f"| {label} | {raw_match:.7f} | {decoded_match:.7f} |")
display(Markdown("\n".join(rows)))
print("Four-spot no-inverse circuit:", schelling_four_spots())

| Angles (A, B) | Without inverse | With inverse |
|---|---:|---:|
| 0, 0 | 1.0000000 | 1.0000000 |
| pi/6, pi/6 | 1.0000000 | 1.0000000 |
| pi/6, 0 | 0.9330127 | 1.0000000 |
| pi/2, 0 | 0.5000000 | 1.0000000 |

Four-spot no-inverse circuit: {'match_rate': 0.9999999999999996, 'unitary_gates': 4, 'cx_count': 2, 'measurement_operations': 4}


The manuscript's sampled no-inverse values 0.936 and 0.487 are plausible
around the exact expectations 0.9330127 and 0.5. The archived pi/6 run prints
0.934, so it does not establish the provenance of 0.936. The original quantum
demos use 1,000 shots; the paper states 10,000.

The four-spot circuit has four unitary gates and four measurements before
adding the inverse. Its eight recorded operations are not eight single-qubit
gates plus two CX gates.

### Compare the independent classical baseline

Classical choice probabilities and quantum rotation angles have different
meanings. A chosen mapping between them must be stated; matching table rows
alone does not ensure matched marginal distributions.

In [6]:
rows = ["| Classical probabilities | Exact match probability |", "|---|---:|"]
for a, b in [(0.5, 0.5), (0.7, 0.7), (0.9, 0.9), (0.8, 0.2), (1, 0.2)]:
    rows.append(f"| {a}, {b} | {classical_match(a, b):.3f} |")
display(Markdown("\n".join(rows)))
print("Independent uniform four-spot baseline: 0.250")

| Classical probabilities | Exact match probability |
|---|---:|
| 0.5, 0.5 | 0.500 |
| 0.7, 0.7 | 0.580 |
| 0.9, 0.9 | 0.820 |
| 0.8, 0.2 | 0.320 |
| 1, 0.2 | 0.200 |

Independent uniform four-spot baseline: 0.250


## What aligns, and what needs correction

- The final grids, many saved cooperative PD outputs, and most Schelling sampled
  rates can be traced to the original notebooks.
- Those PD outputs do not survive the corrected Nash test. Matching the paper's
  interpretation requires a manuscript correction, not a cosmetic code edit.
- The maintained methods specify the actual gate order, player mapping, payoff
  rules, sampling budgets, and numerical tolerances.
- Historical runtimes are retained as observations with their scope. No speedup,
  noise tolerance, or real-world operational benefit is established by these checks.

See [Paper and code](../docs/paper-comparison.md) for the exact page references
and proposed correction wording. The
[original archive](../archive/README.md) preserves the historical work unchanged.